In [ ]:
import pandas as pd
import re
import json
import time
from tqdm import tqdm
from dotenv import load_dotenv
import os
from groq import Groq

load_dotenv()

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)

MODEL = "llama-3.1-8b-instant"
SLEEP_SECONDS = 1

In [33]:
INPUT_PATH = "/Users/barkhaanand/Desktop/AdviksFolder/PROJECTS/vote_your_way/data/prs_datasets/prs_karnataka_bills_processed_partial.csv"

OUTPUT_PATH = "/Users/barkhaanand/Desktop/AdviksFolder/PROJECTS/vote_your_way/data/prs_datasets/prs_karnataka_bills_with_actions.csv"

df = pd.read_csv(INPUT_PATH)

print("Total rows:", len(df))

df.head(2)

Total rows: 180


,title,year,state,pdf_url,bill_text
0,The Karnataka Motor Vehicles Taxation (Amendme...,2023,Karnataka,https://prsindia.org/files/bills_acts/bills_st...,KARNATAKA LEGISLATIVE ASSEMBLY\nSIXTEENTH LEGI...
1,The Karnataka Prohibition of Violence against ...,2023,Karnataka,https://prsindia.org/bills/states/the-karnatak...,NaN


In [34]:
def clean_text(text):

    if not isinstance(text, str):

        return ""

    # Remove encoding noise

    text = re.sub(r"\(cid:\d+\)", " ", text)
    # Remove excessive whitespace

    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [35]:
sample_text = clean_text(df["bill_text"].iloc[0])

print("Length:", len(sample_text))

print(sample_text[:500])

Length: 6685
KARNATAKA LEGISLATIVE ASSEMBLY SIXTEENTH LEGISLATIVE ASSEMBLY FIRST SESSION (Adjourned Meeting) THE KARNATAKA MOTOR VEHICLES TAXATION (AMENDMENT) BILL, 2023 (LA Bill No. 09 of 2023) A Bill further to amend the Karnataka Motor Vehicles Taxation Act, 1957. Whereas it is expedient further to amend the Karnataka Motor Vehicles Taxation Act, 1957 (Karnataka Act 35 of 1957) for the purpose hereinafter appearing: Be it enacted by the Karnataka State Legislature in the seventy fourth year of the Republi


In [36]:
def chunk_text(text, max_words=400):

    words = text.split()

    chunks = []

    

    for i in range(0, len(words), max_words):

        chunk = " ".join(words[i:i+max_words])

        chunks.append(chunk)

    

    return chunks

In [37]:
chunks = chunk_text(sample_text)

print("Total chunks:", len(chunks))
print("First chunk:", chunks[0][:300])

Total chunks: 4
First chunk: KARNATAKA LEGISLATIVE ASSEMBLY SIXTEENTH LEGISLATIVE ASSEMBLY FIRST SESSION (Adjourned Meeting) THE KARNATAKA MOTOR VEHICLES TAXATION (AMENDMENT) BILL, 2023 (LA Bill No. 09 of 2023) A Bill further to amend the Karnataka Motor Vehicles Taxation Act, 1957. Whereas it is expedient further to amend the 


In [38]:
def build_prompt(chunk):

    return f"""

You are analyzing a government bill.

Your task is to extract a SMALL set of HIGH-LEVEL policy actions.

### VERY IMPORTANT RULES:

1. DO NOT extract repetitive or similar actions

2. GROUP similar changes into ONE single action

3. DO NOT list variations (like different years, percentages, slabs separately)

4. Focus on BIG policy changes, not detailed values

5. Keep total outputs between 5 to 10 actions MAX

6. Every action MUST be directly traceable to the text
7. Do NOT reinterpret numerical tables as "increase" or "decrease"
8. If the text defines rates or percentages, describe it as:
  "Define tax rates..." or "Specify tax slabs..."
9. NEVER use words like:
  - increase
  - decrease
  - reduce
UNLESS those exact words appear in the text
### Think like this:

Instead of:

- "Increase tax for 2 years"

- "Increase tax for 3 years"

→ Write:

- "Revise tax structure based on vehicle age"

### Output must:

- Be concise

- Be policy-level

- Be non-redundant

### Output format:

Return ONLY a JSON array of strings.

### Text:

{chunk}

"""

In [39]:
def parse_json_array(text):

    try:

        cleaned = text.replace("```json", "").replace("```", "").strip()

        match = re.search(r"\[.*\]", cleaned, re.DOTALL)

        if not match:

            return []

        data = json.loads(match.group())

        if isinstance(data, list):

            return [str(x).strip() for x in data if str(x).strip()]

        return []

    except:

        return []

In [40]:
def consolidate_actions(actions):
    if not actions:
        return []

    prompt = f"""
You are given a list of policy actions.

Your job:
- Remove duplicates
- Merge similar actions
- Keep only 5–10 high-level actions
- Remove any action that uses inferred words like:
  "increase", "reduce", "waive", unless explicitly present
- Prefer neutral wording like:
  "define", "specify", "introduce", "amend"
### STRICT RULES:
- Return ONLY a JSON array
- DO NOT write code
- DO NOT include ``` or python
- DO NOT include explanations
- Output must start with [ and end with ]

Actions:
{actions[:20]}
"""

    try:
        response = client.chat.completions.create(
            model=MODEL,
            temperature=0,
            messages=[{"role": "user", "content": prompt}],
        )

        raw = response.choices[0].message.content
        print("Raw LLM output:", raw)

        parsed = parse_json_array(raw)

        if not parsed:
            return list(set(actions))  # fallback

        return parsed

    except Exception as e:
        print("Consolidation error:", e)
        return list(set(actions))

In [41]:
def extract_actions_from_text(text):
    print("Function started")
    chunks = chunk_text(text)
    all_actions = []

    for chunk in chunks:
        try:
            prompt = build_prompt(chunk)

            response = client.chat.completions.create(
                model=MODEL,
                temperature=0,
                messages=[{"role": "user", "content": prompt}],
            )

            output = response.choices[0].message.content
            actions = parse_json_array(output)

            all_actions.extend(actions)

            time.sleep(SLEEP_SECONDS)

        except Exception as e:
            print("Error:", e)

    # 🔥 IMPORTANT: second pass consolidation
    final_actions = consolidate_actions(all_actions)

    print(final_actions)   # for testing
    return final_actions

In [ ]:
actions_list = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    text = clean_text(row.get("bill_text", ""))

    

    if len(text) < 100:

        actions_list.append([])

        continue

    actions = extract_actions_from_text(text)

    actions_list.append(actions)

In [ ]:
df["actions"] = actions_list

df.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)

In [ ]:
df[["title", "actions"]].head(5)